Check out my data preprocessing notebook: https://www.kaggle.com/code/rm1000/holistic-age-prediction-data-preprocessing

Check out my model training notebook: https://www.kaggle.com/code/rm1000/holistic-age-prediction-model-selection

Check out my top models notebook: https://www.kaggle.com/code/rm1000/holistic-age-prediction-top-models

In [1]:
import io
import re
import gc
import csv
import gzip
import requests
import textwrap
import subprocess
import numpy as np
import pandas as pd
import urllib.request
!pip install pytorch_tabnet
from sklearn.svm import SVR
from scipy.stats import pearsonr
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from IPython.display import clear_output
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix, f1_score, roc_auc_score
clear_output()

In [2]:
cpg_path = "/kaggle/input/datasets/rm1000/holistic-age-prediction-using-dna-methylation-data/selected_cpgs.txt"
with open(cpg_path, "r") as f:
    cpg_data = [line.strip() for line in f if line.strip()]
print("CpGs Selected:", len(cpg_data))

CpGs Selected: 4872


In [3]:
# 5 min
URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE132nnn/GSE132203/suppl/GSE132203_Geo_Submission_GTPEpic.csv.gz"
OUT_PATH = "/kaggle/working/GSE132203_filtered_5kCpGs_BETA.csv.gz"
cpg_keep = set(cpg_data)
with requests.get(URL, stream=True) as r:
    r.raise_for_status()
    gz_in = gzip.GzipFile(fileobj=r.raw)
    text_in = io.TextIOWrapper(gz_in, encoding="utf-8", newline="")
    reader = csv.reader(text_in)
    header = next(reader)
    keep_idx = [0] + list(range(1, len(header), 2))
    out_header = ["CpG"] + [header[i] for i in keep_idx[1:]]
    kept_rows = 0
    with gzip.open(OUT_PATH, "wt", encoding="utf-8", newline="") as f_out:
        writer = csv.writer(f_out)
        writer.writerow(out_header)
        for row in reader:
            if not row:
                continue
            cg = row[0].strip()
            if cg in cpg_keep:
                writer.writerow([row[i] for i in keep_idx])
                kept_rows += 1
print("Wrote to:", OUT_PATH)
print("CpG rows found:", kept_rows)
print("Requested CpGs:", len(cpg_keep))
print("Missing CpGs:", len(cpg_keep) - kept_rows)
print("Total Samples:", len(keep_idx) - 1)

Wrote to: /kaggle/working/GSE132203_filtered_5kCpGs_BETA.csv.gz
CpG rows found: 4317
Requested CpGs: 4872
Missing CpGs: 555
Total Samples: 795


In [4]:
df = pd.read_csv(OUT_PATH, compression="gzip")
df_t = df.set_index("CpG").T
print("Transposed shape (samples x CpGs):", df_t.shape)
X_epic = df_t.to_numpy(dtype="float32")
sample_ids = df_t.index.to_list()
df_clean = df_t.dropna(axis=1, how='any')
df_t.to_csv("/kaggle/working/GSE132203_filtered.csv", index=False)

Transposed shape (samples x CpGs): (795, 4317)


In [5]:
def parse_geo_soft_family(soft_gz_path):
    records = []
    cur = None
    def flush():
        nonlocal cur
        if cur is not None and cur.get("gsm"):
            records.append(cur)
        cur = None
    with gzip.open(soft_gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("^SAMPLE ="):
                flush()
                cur = {"gsm": line.split("=", 1)[1].strip()}
                continue
            if cur is None:
                continue
            if line.startswith("!Sample_title ="):
                cur["sample_title"] = line.split("=", 1)[1].strip().strip('"')
                continue
            if line.startswith("!Sample_characteristics_ch1 ="):
                val = line.split("=", 1)[1].strip().strip('"')
                parts = [p.strip() for p in val.split(";")]
                for p in parts:
                    if ":" not in p:
                        continue
                    k, v = p.split(":", 1)
                    k = k.strip().lower()
                    v = v.strip()
                    if k == "age":
                        m = NUM_RE.search(v)
                        if m:
                            age_f = float(m.group(0))
                            if 0 < age_f < 120:
                                cur["age"] = int(round(age_f))
                    elif k in {"sex", "gender"}:
                        cur["sex"] = v
                    elif k == "race":
                        cur["race"] = v
                    elif k == "age acceleration":
                        m = NUM_RE.search(v)
                        if m:
                            cur["age_acceleration"] = float(m.group(0))
    flush()
    df = pd.DataFrame(records)
    return df

In [6]:
SOFT_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE132nnn/GSE132203/soft/GSE132203_family.soft.gz"
SOFT_GZ  = "/kaggle/working/GSE132203_family.soft.gz"
r = requests.get(SOFT_URL, stream=True)
r.raise_for_status()
with open(SOFT_GZ, "wb") as f:
    for chunk in r.iter_content(chunk_size=1024*1024):
        if chunk:
            f.write(chunk)
NUM_RE = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
meta_df = parse_geo_soft_family(SOFT_GZ)
display(meta_df.head())
if "age" in meta_df.columns:
    print("Missing Age:", meta_df["age"].isna().sum())
    print("Age Range:", meta_df["age"].min(), "to", meta_df["age"].max())

,gsm,sample_title,sex,age,race,age_acceleration
0,GSM3853125,200928190033_R01C01,Male,49,AA,-4.094808
1,GSM3853126,200928190033_R02C01,Male,21,AA,-4.121570
2,GSM3853127,200928190033_R03C01,Male,44,AA,3.193555
3,GSM3853128,200928190033_R04C01,Male,51,AA,4.723521
4,GSM3853129,200928190033_R05C01,Male,52,Other,0.015870


Missing Age: 0
Age Range: 18 to 76


In [7]:
X = np.array(df_t) # shape (795, 4317)
Xc = np.array(df_clean) # shape (795, 2421)
y = np.array(meta_df["age"]) # shape (795,)
official = np.array(meta_df["age_acceleration"]) # shape (795,)

In [8]:
X_train,X_test,Xc_train,Xc_test,y_train,y_test,official_train,official_test = train_test_split(X,Xc,y,official,test_size=0.25,random_state=42)
gc.collect()
clear_output()

In [9]:
# 3.038
lgbm_model = LGBMRegressor(n_estimators=1000,learning_rate=0.05,min_child_samples=7,num_leaves=15,colsample_bytree=0.5,force_col_wise=True)
lgbm_model.fit(X_train, y_train)
lgbm_preds = lgbm_model.predict(X_test)
lgbm_mae = mean_absolute_error(lgbm_preds,y_test)
print(round(lgbm_mae,3))

[LightGBM] [Info] Total Bins 859682
[LightGBM] [Info] Number of data points in the train set: 596, number of used features: 4317
[LightGBM] [Info] Start training from score 42.684564
3.038


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [10]:
residuals = lgbm_preds-y_test
r, p = pearsonr(residuals, official_test)
print("r =", round(r,3))
print("p =", round(p,3))

r = 0.145
p = 0.041
